# Pluralize a list of Dutch words with Python

This notebook follows the KeesTalksTech example. It reads Dutch and English words from a CSV file, applies explicit overrides, uses `dutch-pluralizer` for Dutch words, and falls back to `inflect` for English words.

In [1]:
input_csv_file_name = "data/test.csv"
input_csv_word_field = 1
input_csv_has_header = True
filter_text = ""

ending_overrides = {
    "mode": "mode",
    "mom": "moms",
    "cut": "cuts",
    "dress": "dresses",
    "scrub": "scrubs",
    "ing": "ing",
    "wax": "wax",
    "olie": "oliën",
    "otion": "otions",
    "speelgoed": "speelgoed",
    "ondergoed": "ondergoed",
    "make-up": "make-up"
}

full_overrides = {
    "mascara": {"singular": "mascara", "plural": "mascara's"},
    "beauty": {"singular": "beauty product", "plural": "beauty producten"},
    "stoppen met roken": {
        "singular": "stoppen met roken",
        "plural": "stoppen met roken",
    },
    "spieren en gewrichten": {
        "singular": "spieren en gewrichten",
        "plural": "spieren en gewrichten",
    },
    "scheren & ontharen": {
        "singular": "scheren & ontharen",
        "plural": "scheren & ontharen",
    },
}

In [2]:
output_csv_file_name = "data/test_output.csv"

In [3]:
from pathlib import Path
import subprocess
import sys

project_root = Path.cwd()
if not (project_root / "dutch_pluralizer").exists():
    project_root = project_root.parent

pluralizer_source = "repo"
pluralizer_version = "0.0.43"

if pluralizer_source == "repo":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(project_root)],
        check=True,
    )
elif pluralizer_source == "pip":
    package = "dutch-pluralizer"
    if pluralizer_version:
        package += f"=={pluralizer_version}"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", package],
        check=True,
    )
    sys.path = [
        path for path in sys.path
        if Path(path or ".").resolve() != project_root.resolve()
    ]
else:
    raise ValueError("pluralizer_source must be 'repo' or 'pip'")



Obtaining file:///workspaces/dutch-pluralizer-py
  Installing build dependencies: started


  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started


  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started


  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started


  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for dutch-pluralizer (pyproject.toml): started


  Building editable for dutch-pluralizer (pyproject.toml): finished with status 'done'
  Created wheel for dutch-pluralizer: filename=dutch_pluralizer-0.0.43-0.editable-py3-none-any.whl size=6193 sha256=d7d79736d672c1b4da0c26fe435e67e7b940d13ac7f2bce6b97ca1de9dd8cfaa
  Stored in directory: /tmp/pip-ephem-wheel-cache-j3talb5r/wheels/63/32/76/854a2e422dc42441bbf1730d3bfca680526d8ae1446e0164a3
Successfully built dutch-pluralizer
  Attempting uninstall: dutch-pluralizer
    Found existing installation: dutch-pluralizer 0.0.43
    Uninstalling dutch-pluralizer-0.0.43:


      Successfully uninstalled dutch-pluralizer-0.0.43


In [4]:
from importlib.metadata import version as installed_package_version

import inflect
import pandas as pd
from IPython.display import display

from dutch_pluralizer import NounEndingMap, Pluralizer
from dutch_pluralizer.speller import ensure_hunspell_nl


In [5]:
input_df = pd.read_csv(
    input_csv_file_name,
    header=0 if input_csv_has_header else None,
    dtype=str,
)
word_values = input_df.iloc[:, input_csv_word_field].fillna("").str.strip()
selected_rows = input_df.loc[
    word_values.ne("")
    & word_values.map(lambda value: not filter_text or filter_text in value)
]
words = (
    selected_rows.iloc[:, input_csv_word_field]
    .tolist()
)

print(f"Loaded {len(words)} words.")
print(f"Sample words: {words[:10]}")

Loaded 21 words.
Sample words: ['kaas', 'auto', 'boek', 'mascara', 'beauty', 'speelgoed', 'dress', 'mode', 'huizen', 'mom']


## Language tools

In [6]:
dutch_speller = ensure_hunspell_nl()
ending_overrides_map = NounEndingMap(ending_overrides)
dutch_pluralizer = Pluralizer(dutch_speller)
dutch_pluralizer.add_ending_overrides(ending_overrides_map)
english_inflector = inflect.engine()

## Determine plural forms

The order matters: full overrides take precedence, followed by the Dutch pluralizer, existing plural detection, ending overrides, and finally the English fallback.

In [7]:
def pluralize_word(word):
    if word in full_overrides:
        plural = full_overrides[word]["plural"]
        source = "full_override"
    else:
        plural = dutch_pluralizer.pluralize(word)
        if plural:
            source = "dutch_pluralizer"
        elif dutch_pluralizer.singularize(word):
            plural = word
            source = "unchanged"
        else:
            plural = ending_overrides_map.pluralize_by_endings(word)
            if plural:
                source = "ending_override"
            elif not dutch_speller.spell(word):
                if english_inflector.singular_noun(word):
                    plural = word
                    source = "unchanged"
                else:
                    plural = english_inflector.plural(word)
                    source = "inflect"
            else:
                plural = None
                source = "not_found"

    return {"word": word, "plural": plural, "source": source}


In [8]:
plural_results = selected_rows.iloc[:, input_csv_word_field].apply(pluralize_word).apply(pd.Series)
plural_df = pd.concat(
    [selected_rows[["id"]], plural_results[["word", "plural", "source"]]],
    axis=1,
)
display(plural_df)

,id,word,plural,source
0,11223,kaas,kazen,dutch_pluralizer
1,23454,auto,auto's,dutch_pluralizer
2,54354,boek,boeken,dutch_pluralizer
3,12345,mascara,mascara's,full_override
4,93844,beauty,beauty producten,full_override
5,29420,speelgoed,speelgoed,dutch_pluralizer
6,40229,dress,dresses,ending_override
7,92113,mode,mode,dutch_pluralizer
8,29223,huizen,huizen,unchanged
9,74821,mom,moms,ending_override


## Determine singular forms

Singular words remain unchanged; plural words are reduced to their singular form.

In [9]:
def singularize_word(word):
    if word in full_overrides:
        singular = full_overrides[word]["singular"]
        source = "full_override"
    else:
        singular = ending_overrides_map.singularize_by_endings(word)
        if singular:
            source = "ending_override"
        else:
            plural = ending_overrides_map.pluralize_by_endings(word)
            if plural:
                singular = word
                source = "ending_override"
            else:
                singular = dutch_pluralizer.singularize(word)
                if singular:
                    source = "dutch_pluralizer"
                else:
                    if not dutch_speller.spell(word):
                        singular = english_inflector.singular_noun(word)
                        source = "inflect" if singular else "unchanged"
                    else:
                        source = "unchanged"
        singular = singular or word

    return {"word": word, "singular": singular, "source": source}

In [10]:
singular_results = selected_rows.iloc[:, input_csv_word_field].apply(singularize_word).apply(pd.Series)
singular_df = pd.concat(
    [selected_rows[["id"]], singular_results[["word", "singular", "source"]]],
    axis=1,
)
display(singular_df)

,id,word,singular,source
0,11223,kaas,kaas,unchanged
1,23454,auto,auto,unchanged
2,54354,boek,boek,unchanged
3,12345,mascara,mascara,full_override
4,93844,beauty,beauty product,full_override
5,29420,speelgoed,speelgoed,ending_override
6,40229,dress,dress,ending_override
7,92113,mode,mode,ending_override
8,29223,huizen,huis,dutch_pluralizer
9,74821,mom,mom,ending_override


## Combine and export results

In [11]:
output_df = selected_rows.merge(
    plural_df[["id", "plural", "source"]].rename(
        columns={"source": "plural_source"}
    ),
    on="id",
    how="left",
    validate="one_to_one",
).merge(
    singular_df[["id", "singular", "source"]].rename(
        columns={"source": "singular_source"}
    ),
    on="id",
    how="left",
    validate="one_to_one",
)

display(output_df)

,id,word,plural,plural_source,singular,singular_source
0,11223,kaas,kazen,dutch_pluralizer,kaas,unchanged
1,23454,auto,auto's,dutch_pluralizer,auto,unchanged
2,54354,boek,boeken,dutch_pluralizer,boek,unchanged
3,12345,mascara,mascara's,full_override,mascara,full_override
4,93844,beauty,beauty producten,full_override,beauty product,full_override
5,29420,speelgoed,speelgoed,dutch_pluralizer,speelgoed,ending_override
6,40229,dress,dresses,ending_override,dress,ending_override
7,92113,mode,mode,dutch_pluralizer,mode,ending_override
8,29223,huizen,huizen,unchanged,huis,dutch_pluralizer
9,74821,mom,moms,ending_override,mom,ending_override


In [12]:
output_df.to_csv(output_csv_file_name, index=False)

found = output_df["plural"].notna().sum()


print(f"Output file: {output_csv_file_name}")
print(f"Words processed: {found} of {len(output_df)}.")

Output file: data/test_output.csv
Words processed: 21 of 21.
